In [1]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv

In [2]:
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv, set_key

load_dotenv()
client = Anthropic()
MODEL = "claude-sonnet-4-6"

## Install Env

In [3]:
env = client.beta.environments.create(
    name="cookbook-data-analyst-env",
    config={
        "type": "cloud",
        "networking": {"type": "unrestricted"},
        "packages": {
            "type": "packages",
            "pip": ["pandas", "plotly"],
        },
    },
)

## Create the agent

In [4]:
ANALYST_SYSTEM_PROMPT = """\
You are a senior data analyst producing a publication-quality report.

## Style
- Professional and precise. Let the data speak with concrete numbers.
- Short paragraphs (2-3 sentences) between charts.
- Lead with the most impactful finding.

## Execution
- Write .py scripts and run them with `python3 script.py`.
- Sample large tables (`nrows=` / `.sample()`) instead of loading everything.
- Sanity-check key metrics before building narrative around them.

## Charts
- Build each chart as its own `go.Figure()`, embed with
  `fig.to_html(include_plotlyjs=False, full_html=False)`, and load plotly
  from the CDN once in <head>.
- Always set `marker_color` and `template='simple_white'`.

## Output
Write a single self-contained `report.html` to /mnt/session/outputs/
with inline CSS, 3+ embedded plotly charts, and a closing section of
actionable recommendations. Confirm "Saved: report.html" when done.
"""

agent = client.beta.agents.create(
    name="cookbook-data-analyst",
    model=MODEL,
    system=ANALYST_SYSTEM_PROMPT,
    tools=[
        {
            "type": "agent_toolset_20260401",
            # default_config applies to every tool in the set;
            # entries in configs override specific tools.
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
            "configs": [
                {"name": "web_search", "enabled": False},
                {"name": "web_fetch", "enabled": False},
            ],
        }
    ],
)

## Upload sample data

In [5]:
DATA_PATH = Path("sample_data/fifa_wc_data.csv")

with DATA_PATH.open("rb") as f:
    dataset = client.beta.files.upload(file=(DATA_PATH.name, f, "text/csv"))

print(f"Uploaded {DATA_PATH.name} ({dataset.size_bytes} bytes) as {dataset.id}")

Uploaded fifa_wc_data.csv (4359 bytes) as file_011Cc594FTeNqpN1P47zJGvb


## Create the session and send the task

In [7]:
MOUNT_PATH = f"/mnt/session/uploads/{DATA_PATH.name}"

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[{"type": "file", "file_id": dataset.id, "mount_path": MOUNT_PATH}],
    title="Sales analysis",
)

ANALYSIS_PROMPT = f"""\
Analyze the fifa wc data in {MOUNT_PATH}.

Columns: team, continent, is_host, goals_scored_last_4y, goals_received_last_4y, wins_last_4y, losses_last_4y, draws_last_4y, world_cup_titles_before, squad_total_market_value_eur, fifa_rank_pre_tournament, fifa_points_pre_tournament, squad_avg_age, world_cup_participations_before, groups_passed_before, round16_before, quarterfinals_before, semifinals_before, finals_before

Focus on team performance in the past and their progression in the past. Produce report.html per your system instructions.
"""

client.beta.sessions.events.send(
    session.id,
    events=[
        {"type": "user.message", "content": [{"type": "text", "text": ANALYSIS_PROMPT}]},
    ],
)
print(f"Session {session.id} running")

Session sesn_015Ybk4d19TcqtUvDPXKon2z running


In [8]:
def wait_for_idle(session_id: str) -> None:
    for ev in client.beta.sessions.events.stream(session_id):
        t = ev.type
        if t == "agent.message":
            for block in ev.content:
                if block.type == "text":
                    text = block.text
                    print(text[:300] + ("..." if len(text) > 300 else ""))
        elif t in ("agent.tool_use", "agent.mcp_tool_use"):
            print(f"  [{ev.name}]")
        elif t == "session.status_idle":
            return
        elif t == "session.status_terminated":
            raise RuntimeError(
                "Session terminated before going idle. "
                f"Trace: https://platform.claude.com/sessions/{session_id}"
            )


wait_for_idle(session.id)

  [bash]
  [bash]
Good, I have a clear picture of the data. Let me now build the full analysis and report.
  [write]
  [bash]
Let me verify the key narrative numbers before finalizing:
  [bash]
I see key corrections needed — narrative errors in title counts, DRI values, and an England edge case where QFs exceed R16s (pre-1986 format difference). Let me fix everything:
  [write]
  [bash]
  [bash]
All numbers verified. **Saved: report.html** ✅

---

Here's a summary of what the report covers across its 6 interactive charts:

---

### Report Structure

**6 Embedded Plotly Charts:**

| # | Chart | Key Insight |
|---|-------|-------------|
| 1 | **Historical Stage-Reach Overlay** (top 20) | Germ...


In [9]:
outputs = client.beta.files.list(scope_id=session.id, betas=["managed-agents-2026-04-01"])
for f in outputs.data:
    print(f.filename, f.size_bytes)

# The list also includes the mounted input CSV; pick out the report.
report = next((f for f in outputs.data if f.filename == "report.html"), None)
if report is None:
    raise RuntimeError(f"report.html not found. Files: {[f.filename for f in outputs.data]}")
content = client.beta.files.download(report.id)
Path("report.html").write_bytes(content.read())
print("Downloaded report.html")

report.html 92665
fifa_wc_data.csv 4359
Downloaded report.html


In [10]:
client.beta.sessions.archive(session.id)

set_key(".env", "ANALYST_ENV_ID", env.id)
set_key(".env", "ANALYST_AGENT_ID", agent.id)
set_key(".env", "ANALYST_AGENT_VERSION", str(agent.version))
print("Saved ANALYST_ENV_ID, ANALYST_AGENT_ID, ANALYST_AGENT_VERSION to .env")

Saved ANALYST_ENV_ID, ANALYST_AGENT_ID, ANALYST_AGENT_VERSION to .env
